# CO-only equilibrium conversion

Run from this folder with the qiao kernel. Read the parent `encoded_dataset.xlsx` and the local copy of the original AramcoMech 3.0 thermodynamic YAML. Evaluate TP equilibrium for one ideal-gas phase with a binary H₂/CO₂ inlet at the reported ratio. The input labels indicate some N₂- or Ar-diluted feeds, and the calculation uses the original binary-feed convention.

The equilibrium conversion is calculated from the fraction of gas-phase carbon remaining in CO₂, with two carbon atoms per DME or methyl formate molecule. ATE is the observed conversion divided by the calculated equilibrium conversion. The Cantera solver settings and the input hashes are written to the result workbook.

[Cantera ThermoPhase API](https://cantera.org/3.0/sphinx/html/cython/thermo.html) · [AramcoMech 3.0 thermodynamic data](https://universityofgalway.ie/combustionchemistrycentre/mechanismdownloads/)

In [ ]:
from functools import lru_cache
from hashlib import sha256
from pathlib import Path
import cantera as ct
import numpy as np
import pandas as pd
from IPython.display import display

In [ ]:


dataset_path = Path("derived") / "dataset.xlsx"  # Path to the required dataset workbook.
thermo_path = Path("thermo") / "AramcoMech3.0.yaml"  # Path to the Cantera mechanism and thermodynamic data.
output_dir = Path("equilibrium_results")  # Folder for generated results.
output_dir.mkdir(exist_ok=True)
point_id = "Data point number"  # Unique experimental-record identifier column.
condition_columns = ["T", "P", "H2/CO2"]  
source_columns = [ 
    point_id, "Catalyst number", "Reference DOI", *condition_columns, "Byproduct",
    "Feed Composition", "Con", "Sel", "EqCon", "Con/EqCon",
]
base_species = ("CO2", "H2", "CH3OH", "H2O")  # Base species.
elements = ("C", "H", "O")  
eq_tolerance = 1e-6 
ate_tolerance = 1e-7 
solver_rtol = 1e-9 

data = pd.read_excel(dataset_path, sheet_name="data", usecols=source_columns, engine="openpyxl")  # Input records for this analysis.
if data[point_id].isna().any() or not data[point_id].is_unique:
    raise ValueError("Data point numbers must be present and unique.")
numeric = [*condition_columns, "Con", "Sel", "EqCon", "Con/EqCon"]  
data[numeric] = data[numeric].apply(pd.to_numeric)
if not np.isfinite(data[numeric].to_numpy(dtype=float)).all():
    raise ValueError("Conditions, observed outcomes and reference equilibrium values must be finite.")
if data[[*condition_columns, "EqCon"]].le(0).any().any():
    raise ValueError("Temperature, pressure, feed ratio and stored equilibrium conversion must be positive.")
if not np.allclose(data["Con"] / data["EqCon"], data["Con/EqCon"], atol=ate_tolerance, rtol=0):
    raise ValueError("The stored ATE column does not agree with Con divided by the stored EqCon.")

species_db = {  # Species db.
    item.name: item for item in ct.Species.list_from_yaml(thermo_path.read_text(encoding="utf-8-sig"), "species")  
}


@lru_cache(maxsize=None)
def gas_phase(names):  # Construct an ideal-gas phase for the allowed species.
    species = [species_db[name] for name in names]  # Chemical species admitted to the equilibrium calculation. Current item name.
    return ct.Solution(thermo="ideal-gas", species=species)


@lru_cache(maxsize=None)
def equilibrium(temp, pressure_mpa, h2_co2, extras):  # Return the constant-temperature, constant-pressure equilibrium result.
    names = base_species + extras  
    gas = gas_phase(names)  # Cantera ideal-gas mixture.
    if not gas.min_temp <= temp <= gas.max_temp:
        raise ValueError(f"Temperature {temp} K lies outside the thermodynamic range of {names}.")
    pressure_pa = pressure_mpa * 1e6  # Pressure pa.
    gas.TPX = temp, pressure_pa, {"CO2": 1, "H2": h2_co2}
    initial = np.array([gas.elemental_mass_fraction(element) for element in elements])  
    gas.equilibrate("TP", solver="auto", rtol=solver_rtol)
    final = np.array([gas.elemental_mass_fraction(element) for element in elements])  
    elemental_error = float(np.max(np.abs(final - initial)))  
    if elemental_error > 1e-8:
        raise ValueError(f"Elemental mass-fraction error {elemental_error:.3g} for {temp}, {pressure_mpa}, {h2_co2}.")
    if not np.isclose(gas.T, temp, rtol=1e-8) or not np.isclose(gas.P, pressure_pa, rtol=1e-8):
        raise ValueError("The solution departed from the prescribed temperature or pressure.")
    carbon_counts = np.array([gas.n_atoms(index, "C") for index in range(gas.n_species)])  # Carbon counts. Current position or index label.
    gas_carbon = float(gas.X @ carbon_counts)  
    conversion = 100 * (1 - gas["CO2"].X[0] / gas_carbon)  
    if not np.isfinite(conversion) or not 0 < conversion <= 100:
        raise ValueError(f"Invalid calculated equilibrium conversion {conversion}.")
    return conversion, elemental_error


def calculate_case(extras_by_row):  # Calculate equilibrium conversion for one record and species set.
    temperatures, pressures, ratios = (data[column] for column in condition_columns)  # Temperatures; Pressures; Ratios. Current column name.
    rows = []  # Rows accumulated for the result table.
    for point, temp, pressure, ratio, extras in zip(data[point_id], temperatures, pressures, ratios, extras_by_row, strict=True):  
        try:
            conversion, error = equilibrium(temp, pressure, ratio, extras)  
        except (ct.CanteraError, ValueError, KeyError) as exc:  
            raise RuntimeError(f"Equilibrium calculation failed for data point {point}, extras={extras}.") from exc
        rows.append((conversion, error, ", ".join(base_species + extras)))
    result = pd.DataFrame(rows, columns=["EqCon (%)", "Max element mass fraction error", "Gas species"], index=data.index)  
    result["ATE"] = data["Con"] / result["EqCon (%)"]
    result["EqCon difference (pp)"] = result["EqCon (%)"] - data["EqCon"]
    result["ATE difference"] = result["ATE"] - data["Con/EqCon"]
    result["EqCon reproduced"] = result["EqCon difference (pp)"].abs().le(eq_tolerance)
    result["ATE reproduced"] = result["ATE difference"].abs().le(ate_tolerance)
    return result


def case_summary(result):  # Summarize conversion reproduction and ATE for one equilibrium case.
    error = result["EqCon difference (pp)"].abs()  
    return {
        "Records": len(result),
        "EqCon reproduced": int(result["EqCon reproduced"].sum()),
        "ATE reproduced": int(result["ATE reproduced"].sum()),
        "EqCon MAE (pp)": error.mean(),
        "EqCon max absolute difference (pp)": error.max(),
        "Mean absolute ATE difference": result["ATE difference"].abs().mean(),
        "ATE > 1": int(result["ATE"].gt(1).sum()),
        "Max element mass fraction error": result["Max element mass fraction error"].max(),
    }


metadata = pd.DataFrame([  
    ("Dataset", dataset_path.name),
    ("Dataset SHA-256", sha256(dataset_path.read_bytes()).hexdigest()),
    ("Thermodynamic data", thermo_path.name),
    ("Thermodynamic data SHA-256", sha256(thermo_path.read_bytes()).hexdigest()),
    ("Cantera version", ct.__version__),
    ("Phase", "Single ideal gas with the listed gas species"),
    ("Temperature and pressure", "T in K, P in MPa converted to Pa. Isothermal, isobaric equilibrium."),
    ("Feed assumption", "Binary H2/CO2 at the reported ratio. N2/Ar dilution is not quantified in the encoded input."),
    ("Solver", f"equilibrate('TP', solver='auto', rtol={solver_rtol:g})"),
    ("Conversion", "100 * [1 - x(CO2) / sum_i(n_C,i * x_i)] for this CO2-only carbon feed"),
    ("ATE", "Observed conversion (%) / equilibrium conversion (%)"),
    ("EqCon reproduction tolerance (pp)", eq_tolerance),
    ("ATE reproduction tolerance", ate_tolerance),
], columns=["Setting", "Value"])

In [ ]:
co = calculate_case([("CO",)] * len(data))  
summary = pd.DataFrame([{"Scenario": "CO only", **case_summary(co)}])  
results = data.join(co.add_prefix("Recalculated "))  
feed_counts = data["Feed Composition"].value_counts(dropna=False).rename("N").reset_index()  
with pd.ExcelWriter(output_dir / "eqcon_co.xlsx", engine="openpyxl") as writer:  
    results.to_excel(writer, sheet_name="by_point", index=False)
    summary.to_excel(writer, sheet_name="reproduction", index=False)
    feed_counts.to_excel(writer, sheet_name="feed_labels", index=False)
    metadata.to_excel(writer, sheet_name="method", index=False)
display(summary)